# MetaSAG Workflow Simple Test Notebook

This notebook is a lightweight test file for the MetaSAG workflow described in `MetaSAG/README.md`. It uses the bundled example data to demonstrate and verify the main analysis process, from droplet read splitting and low-quality cell filtering to taxonomic assignment, genome/tree analysis, strain resolution, horizontal gene transfer analysis, and HUMAnN functional pathway analysis, before applying MetaSAG to real microbial SAG droplet sequencing data.

Because the example dataset is intentionally small, this notebook is intended mainly as a workflow demonstration rather than a full-scale biological analysis. Some downstream steps may produce limited outputs, empty results, or no test result files, depending on the amount and composition of the available test data.

Please enter the sample name and the result output path. `SampleName` is used as the prefix for test outputs, and `Target_Path` is the directory where all workflow results will be saved. The recommended sample name for this test notebook is `Test`.

In [ ]:
SampleName = "Test"
Target_Path = "Your/Result/Path/"

# Step 1. Distribute the reads in the sample to a file of individual droplets


Please enter the paths to the paired-end FASTQ test files used in Step 1. These files are used to demonstrate read distribution into individual droplets.

In [ ]:
InputFastq = [
    "Example_data/Step_1to5_9_TestData/Test_R1.fastq",
    "Example_data/Step_1to5_9_TestData/Test_R2.fastq"
]

In [ ]:
from MetaSAG import BarcodeDeal as bd
from MetaSAG import BarcodeTagFunc as btf

ResultDir = Target_Path + "Barn/"

TagFunc = btf.make_tag_func(SampleName)

Barcode = bd.Barcode(InputFastq, ResultDir)
Barcode.BarcodeTag(TagFunc=TagFunc)
Barcode.trim()
Barcode.SplitTaggedFastqByCell()

# Step 2. Filter low-quality cells

In [ ]:
from MetaSAG import BCFilter as bcf

ResultDir = Target_Path + "Cell_Filter/"
Filter_path = Target_Path + "Cell_Filter/Filtered_Fastq/"

InputFastq = [
    Target_Path + "Barn/Trim/Test_R1.fastq",
    Target_Path + "Barn/Trim/Test_R2.fastq"
]

obj = bcf.BCFilter(InputFastq, ResultDir)
obj.CellCountStatistic()
obj.getMinReads()
obj.BCMinReads(Filter_path, filter_outname=SampleName)

# Step 3. MetaPhlAn4 annotates the reads and classifies droplets



Please enter the environment names and database paths required for Step 3. `METAPHLAN_ENV` and `METAPHLAN_DB` are used for MetaPhlAn4 annotation, `SPADES_ENV` is used for cell assembly, and `KRAKEN_ENV` and `KRAKEN_DB` are used for identifying potential multi-cell droplets.

In [ ]:
METAPHLAN_ENV = "metaphlan4.1"
METAPHLAN_DB = "/Database/MetaPhIAn4_1/DB_Jun23/"

SPADES_ENV = "spades"
KRAKEN_ENV = "kraken"
KRAKEN_DB = "/Database/k2_standard_20230605/"

In [ ]:
from MetaSAG import MetaPhlAnAsign as mpa
import pandas as pd

MPOut = Target_Path + "MetaPhlAnAsign/annotation_file/"

mpa.MPAnno(InputFastq, MPOut, SampleName, env=METAPHLAN_ENV, DB=METAPHLAN_DB)

Input_bowtie = Target_Path + "MetaPhlAnAsign/annotation_file/Test_bowtie2"
Result_dir = Target_Path + "MetaPhlAnAsign/MPAsign/"

obj = mpa.MPBowtie(Input_bowtie, Result_dir)
obj.CellSGBStatistic()

BC_Count = pd.read_csv(Target_Path + "Cell_Filter/bcread.txt", sep="\t", header=0)

obj.CellAsign(BC_Count)
obj.HostPhage()

CellBarn = Target_Path + "Barn/Cell/"

obj.DoubleCellKraken(CellBarn, env=KRAKEN_ENV, KrakenDB=KRAKEN_DB, ReadsEnd="Pair")
obj.CellAssem(CellBarn,env=SPADES_ENV, ReadsEnd="Pair")

# Step 4. Droplet clustering of potentially unknown species

Note: Step 4 clusters droplets that remain unassigned after the MetaPhlAn4-based annotation in Step 3. It mainly includes: `FilterUnassignedCells()` to select droplets labeled as `UnAsignedCell` from `CellAnno.txt`, `ClusterSAG()` to perform the first round of clustering and assembly for potentially unknown species, and `ClusterBin()` to optionally re-cluster the initial bins based on the first-round results.

Because the Step 1–5 example dataset is intentionally small, it may contain too few unassigned droplets, or the selected droplets may have insufficient sequencing depth. Therefore, this step is included mainly as a workflow demonstration in this notebook. The generated clustering or assembly results may be empty, limited, or not biologically meaningful. For real biological interpretation, users should run this step on their own data with enough unassigned droplets and sufficient read depth.

Please enter the conda environment names required for Step 4. `SpadesEnv` is used to run SPAdes for assembling selected unassigned droplets and droplet bins, and `SourmashEnv` is used to run Sourmash for signature calculation and clustering.

In [ ]:
SpadesEnv = "spades"
SourmashEnv = "sourmash"

In [ ]:
from MetaSAG import UnknownSAG as usag
import os

cellAnno = Target_Path + "MetaPhlAnAsign/MPAsign/CellAnno.txt"
fastqDir = Target_Path + "Barn/Cell/"
outputDir = Target_Path + "UnknownSAG/"
fastqOutputDir = outputDir + "Fastq/"

usag.FilterUnassignedCells(cellAnno, fastqDir, fastqOutputDir)

usag.ClusterSAG(
    fastqOutputDir,
    outputDir,
    ReadsEnd="Pair",
    SpadesEnv=SpadesEnv,
    SourmashEnv=SourmashEnv
)

Round1Dir = os.path.join(outputDir, "Round1")
Round2Dir = os.path.join(outputDir, "Round2")

usag.ClusterBin(
    Round1Dir,
    Round2Dir,
    ReadsEnd="Pair",
    SpadesEnv=SpadesEnv,
    SourmashEnv=SourmashEnv
)

# Step 5. Quality control and integration annotation of assembled genome

Note: Step 5 performs quality control and integrated annotation for assembled genomes generated from Step 3. It mainly includes: `FastaQC1()` to remove short contigs, `FastaQC2()` to classify genomes into `Pass`, `Bandage`, or `Abandon` based on CheckM results, and `Summary()` to integrate MetaPhlAn4 classification, CheckM quality assessment, and GTDB-TK taxonomic annotation results.

Because the Step 1–5 example dataset is intentionally small, Step 3 may not assemble enough genomes for a complete Step 5 test. Therefore, this step is included mainly as a workflow demonstration in this notebook. The `BinFasta/` directory may be empty or contain only a few genomes, `BinFastaQC2/Pass/` may not retain valid downstream genomes, and `summary.txt` may be empty or not generated with informative results. For real biological analysis, users should run this step with sufficiently assembled genomes and prepare the corresponding CheckM and GTDB-TK result files in advance.

In [ ]:
from MetaSAG import BinQCAnno as bqa

FastaDir = Target_Path + "MetaPhlAnAsign/MPAsign/BinFasta/"
FastaOut = Target_Path + "Bin_QC/BinFastaQC1/"

bqa.FastaQC1(FastaDir, FastaOut)

In [ ]:
FastaDir = Target_Path + "Bin_QC/BinFastaQC1/"
FastgDir = Target_Path + "MetaPhlAnAsign/MPAsign/BinFastg/"
CheckmFile = Target_Path + "Bin_QC/BinFastaQC1/check/qa_results"
FastaOut = Target_Path + "Bin_QC/BinFastaQC2/"

bqa.FastaQC2(FastaDir, FastaOut, CheckmFile, FastgDir=FastgDir)

In [ ]:
FastaSGB = Target_Path + "Bin_QC/BinFastaQC2/Pass/FastaSGB.txt"
CheckmFile = Target_Path + "Bin_QC/BinFastaQC1/check/qa_results"
GTDBFile = Target_Path + "Bin_QC/BinFastaQC2/Pass/gtdb_out/gtdbtk.bac120.summary.tsv"
outputSummary = Target_Path + "Bin_QC/summary.txt"

summary = bqa.Summary(FastaSGB, CheckmFile, GTDBFile, outputSummary)

CheckM and GTDB-TK should be run externally by users before executing the corresponding MetaSAG functions. They generate the quality assessment and taxonomic annotation files required by `FastaQC2()` and `Summary()`. The following commands are provided as examples; please modify the conda environment, database path, input directory, and thread number according to your own setup.

In [ ]:
BASE="Your/Result/Path/Bin_QC/BinFastaQC1"

# 1. Run CheckM lineage workflow.
checkm lineage_wf -t 32 -x fasta "$BASE/" "$BASE/check"

# 2. Export the QA result table.
checkm qa -o 2 --tab_table -f "$BASE/check/qa_results" "$BASE/check/lineage.ms" "$BASE/check"

In [ ]:
export GTDBTK_DATA_PATH=/Database/GTDBTK-2.1.1_DB

BASE="Your/Result/Path/Bin_QC/BinFastaQC2/Pass"

gtdbtk classify_wf \
    --genome_dir "$BASE/" \
    --out_dir "$BASE/gtdb_out/" \
    --extension fasta \
    --cpus 80 \
    --pplacer_cpus 40 \
    --prefix gtdbtk \
    --force

# Step 6. Build phylogenetic tree.

Please enter the test data path and the conda environment name required for Step 6. `TestData` contains the example genomes and annotation files, and `ANVIO_ENV` is used to run anvi’o for phylogenetic tree construction.

In [ ]:
TestData = "/Example_data/Step_6_8_TestData/"
ANVIO_ENV="anvio-7.1"

In [ ]:
from MetaSAG import Tree as tree

FastaDir = TestData
CellAnno = TestData + "CellAnno.txt"
Summary = TestData + "summary.txt"

BinAnno = Target_Path + "Tree/BinAnno.txt"
TreeTemp = Target_Path + "Tree/TreeTemp/"
AnnoResult = Target_Path + "Tree/AnnoResult/"

tree.GenerateBinAnno(FastaDir, CellAnno, Summary, BinAnno)
tree.BuildTree(FastaDir, TreeTemp, env=ANVIO_ENV)
tree.itolPlot(BinAnno, AnnoResult)

# Step 7. Species to Strain resolved genomes.

Please enter the test data path and tool parameters required for Step 7. `snap_aligner`, `bcftools`, and `samtools` are executable paths or command names used for read alignment, SNP calling, and BAM sorting, and `SPADES_ENV` is used for strain-level genome assembly.

In [ ]:
TestData = "/Example_data/Step_7_TestData/"

snap_aligner = '/Tools/SNAP/snap-aligner'
bcftools = '/Tools/bcftools/bcftools-1.18/bcftools'
samtools = '/Tools/samtools/samtools-1.17/samtools'
SPADES_ENV = "spades"

In [ ]:
from MetaSAG import SNPStrain as snp

SpeciesName = "SGB8002"
BinDir = Target_Path + "SNPStrain/SingleBin/SGB8002/"
ResultDir = Target_Path + "SNPStrain/SingleBin/SGB8002_Result/"

SingleBin = snp.SingleBin(BinDir, ResultDir, SpeciesName)

FastaDir = TestData
FastqDir = TestData
CellAnno = TestData + "CellAnno.txt"

SingleBin.SetupRunDir(FastaDir, FastqDir, CellAnno, ReadsEnd="Single")

SingleBin.SingleBinPrepare(
    bcftools=bcftools,
    snap_aligner=snap_aligner,
    samtools=samtools,
    ReadsEnd="Single",
    sort_tool="samtools"
)

# Determine the cluster number based on the UMAP and tree plots generated by SingleBinPrepare().
SingleBin.SingleBinSplit(2)

StrainAssemDir = Target_Path + "SNPStrain/SingleBin/StrainAssem/"

SingleBin.StrainAssem(StrainAssemDir=StrainAssemDir, ReadsEnd="Single",env = SPADES_ENV)

# Step 8. Horizontal Gene Transfer.

Please enter the test data path, conda environment names, and database path required for Step 8. `PROKKA_ENV`, `CDHIT_ENV`, and `EMAPPER_ENV` are used for gene prediction, clustering, and functional annotation, and `EMAPPER_DB` is the eggNOG-mapper database path.

In [ ]:
FastaDir = "/Example_data/Step_6_8_TestData/"

PROKKA_ENV = "prokka"
CDHIT_ENV = "base"
EMAPPER_ENV = "eggnog-mapper2"
EMAPPER_DB = "/Database/eggnogDB/"

In [ ]:
from MetaSAG import CellHGT as hgt

TreeAnno = Target_Path + "Tree/BinAnno.txt"
HGTTemp = Target_Path + "HGT/"

obj = hgt.CellHGT(FastaDir, HGTTemp)

obj.SpeciesHGT()
obj.HGTSpeciesPlot(TreeAnno)
obj.HGTSpeciesAnno(
    TreeAnno,
    prokka_env=PROKKA_ENV,
    cdhit_env=CDHIT_ENV,
    emapper_env=EMAPPER_ENV,
    emapper_DB=EMAPPER_DB
)

# Step 9. HUMAnN Path.

Please enter the database path and conda environment names required for Step 9. `DIAMOND_DB` is the UniRef90 DIAMOND database, `DIAMOND_ENV` is used to run DIAMOND, and `HUMANN_ENV` is used to run HUMAnN for pathway analysis.

In [ ]:
DIAMOND_DB="/Database/uniref/uniref90_201901b_full.dmnd"
DIAMOND_ENV= "diamond"

HUMANN_ENV="humann"

In [ ]:
from MetaSAG import HUMAnNPath as hp

FastqDir = Target_Path + "Barn/Trim/Test_R1.fastq"
ResultDir = Target_Path + "HUMAnNPath/"

obj = hp.HP(FastqDir, ResultDir)

obj.Diamond(
    DiamondDB=DIAMOND_DB,
    Diamondenv=DIAMOND_ENV
)

obj.Uniref2Matrix()
obj.SeuratCluster()

cellAnno = Target_Path + "MetaPhlAnAsign/MPAsign/CellAnno.txt"

obj.HUMAnNPath(cellAnno, "SGB", HUMAnNenv=HUMANN_ENV,ExcludeGroups="NoSGB")